In [1]:
import pandas as pd
import requests
import re
import time
from bs4 import BeautifulSoup as bs

# 뉴스 목록, 날짜, id 수집

In [8]:
url1 = "https://fintech.or.kr/web/board/boardContentsList.do"
payload1 = dict(miv_pageNo=1, mode="W", board_id=6)
r1 = requests.post(url1, data=payload1)
soup = bs(r1.content, "lxml")
soup

<html><head><script language="javascript">
$(document).ready(function(){

	// 댓글수 노출여부 확인후 숨기기(미구현)
	
		$(".comment_cnt").hide();
	

	// 답변 노출여부 확인후 숨기기(미구현)
	
		$(".reply_status").hide();
	

	// 모바일 전용 페이지 숨기지
	//$(".mobile_list").hide();

	$(function() {

	});

	$("#searchtxt").keydown(function(key) {
		if (key.keyCode == 13) {
			search();
		}
	});
});
</script>
</head><body><div class="boardlist_top">
<ul class="box">
<li class="select">
<div class="optionbox">
<select id="searchkey" name="searchkey">
<option value="I">전체</option>
<option value="T">제목</option>
<option value="C">내용</option>
</select>
</div>
</li>
<li class="input">
<div class="inpbox"><input class="txt" id="searchtxt" name="searchtxt" placeholder="검색어 입력" title="검색어 입력" type="text" value=""/></div>
</li>
<li class="button">
<button class="btn3 bg_blue" onclick="search();" title="검색" type="button">검색</button>
<!-- <button type="button" class="btn3 bg_gray" title="재검색">재검색</button> -->
</li>
</ul>
</div>
<!--// list_t

In [33]:
len(soup.select("a.txtl"))

15

In [55]:
len(soup.select("td.last"))

15

In [45]:
def text_clean(text):
    return re.sub("[^가-힣a-zA-Z0-9-]", "", text)
    

In [46]:
text_clean(soup.select("td.last")[0].text)

'2025-09-17'

In [47]:
contents_id_list = []
for date, content_list in zip(soup.select("td.last"), soup.select("a.txtl")):
    date = text_clean(date.text)
    contents_id = content_list['href'].split("'")[1]
    contents_id_list.append((date, contents_id))
contents_id_list

[('2025-09-17', 'b4b0b17c318d45158645c81c1472e29a'),
 ('2024-02-20', '51cb0e37c4064f2a985436e2ae3378df'),
 ('2022-12-19', '3c24d05040984c79a7cdfd2c921952eb'),
 ('2022-12-19', '25f4360d5b9847a996828099e8d52ade'),
 ('2022-06-20', '9c16609a99ad4c55914ffa9a7f2d58b7'),
 ('2021-12-03', '6b14460b5adc480aa050c4c535af8b72'),
 ('2021-04-07', 'd4cbbeb3f9b3444cbf1856b48f9f2926'),
 ('2025-10-21', 'f61663288d694d068e33d00054b22595'),
 ('2025-10-20', '760c4ae4b76f449aa23de004e8200bc5'),
 ('2025-10-17', 'ad2eb2c521ab4e9fa32cb00a625cbd69'),
 ('2025-10-16', 'c80032073f994fb4b669d70bb5917c18'),
 ('2025-10-15', 'c579f3baf033478483098dc813b47e1f'),
 ('2025-10-14', 'f7a7f75aa51045209c20eff82dc21e90'),
 ('2025-10-13', 'b45fb58dc8094571a48344594ab19176'),
 ('2025-10-10', 'aeea59af5aea4346a8d53ed2605993e5')]

# 날짜별 뉴스 모음에 들어가 해당 날짜 뉴스 모음 제목과 링크 수집하기

In [68]:
result = {}
for idx, (date, content_id) in enumerate(contents_id_list):
    print(f"{idx+1}/{len(contents_id_list)} {content_id} 수집중", end="\r")
    url2 = "https://fintech.or.kr/web/board/boardContentsView.do"
    payload2 = dict(miv_pageNo=1, mode="W", contents_id=content_id, board_id=6, searchkey="I")
    r2 = requests.post(url2, data=payload2)
    soup2 = bs(r2.content, "lxml")
    
    for td in soup2.select("tbody td"):
        if td.select_one("a") != None and td.select_one("a")['href'] != "https://fintech.or.kr/":
    #         print(td.select_one("a").text)
    #         print(td.select_one("a")['href'])
            result.setdefault("제목", []).append(td.select_one("a").text)
            result.setdefault("원문링크", []).append(td.select_one("a")['href'])
df = pd.DataFrame(result)

In [69]:
df

,제목,원문링크
0,"한국핀테크지원센터, 핀에듀 AI 활용 실무 온라인 교육 과정 5종 신규 개설",https://www.etnews.com/20251017000242
1,[한국핀테크지원센터] 2025 핀테크 전문가(개발자) 과정 ‘FinBoost Aca...,https://fintech.or.kr/web/board/boardContentsV...
2,[한국핀테크지원센터] AICC 기반 금융 서비스 자동화 교육 과정 수강생 모집(~선착순),https://fintech.or.kr/web/board/boardContentsV...
3,[한국핀테크지원센터] (온라인) 핀테크를 통한 금융 AI 트렌드와 혁신 사례 교육생...,https://fintech.or.kr/web/board/boardContentsV...
4,[한국핀테크지원센터] 2025년 핀테크 통번역존 운영(~예산 소진 시까지),https://fintech.or.kr/web/board/boardContentsV...
...,...,...
401,"美 셧다운·금리인하 기조에… 금·은·코인으로, 돈의 대이동",https://n.news.naver.com/mnews/article/008/000...
402,"“AI붐은 버블, 25년전 닷컴 때과 비슷”...IMF·영란은행, 증시 과열에 경고",https://n.news.naver.com/mnews/article/009/000...
403,[디캠프·한국핀테크지원센터] startup OI #금융권-스타트업 우수 협력 사례 ...,https://fintech.or.kr/web/board/boardContentsV...
404,[금융보안원] 「데이터허브」 기업 무료 이용 안내(상시),https://fintech.or.kr/web/board/boardContentsV...


In [70]:
for title in df['제목']:
    print(title)

한국핀테크지원센터, 핀에듀 AI 활용 실무 온라인 교육 과정 5종 신규 개설
[한국핀테크지원센터] 2025 핀테크 전문가(개발자) 과정 ‘FinBoost Academy’참여자 모집(~선착순 마감)
[한국핀테크지원센터] AICC 기반 금융 서비스 자동화 교육 과정 수강생 모집(~선착순)
[한국핀테크지원센터] (온라인) 핀테크를 통한 금융 AI 트렌드와 혁신 사례 교육생 모집(상시)
[한국핀테크지원센터] 2025년 핀테크 통번역존 운영(~예산 소진 시까지)
[한국핀테크지원센터] 2025년 핀테크기업 온라인 채용관 (사람인 saramin) 참여기업 모집 (~예산 소진 시까지)
금융위원장 “금산분리 합리화, 실용적 방안 고민중…핀테크 지분 투자 확대 허용”
스테이블코인 법제화 가시화…금융위 “준비자산은 예금·국채 등으로 100% 보유 의무”
FIU "가상자산거래소 '오더북 공유', 개인정보 불법유출 감독할 것"
금융위원장 "부동산 자금쏠림 개선…필요시 즉각 추가 조치"
이억원 "금융소비자 보호…보이스피싱 금융권 책임 법제화"
금감원, 이번주 ‘홍콩 ELS’ 일부 과태료 결정
렌탈페이, 비영리단체·지자체와 손잡고 ‘상생 결제 문화’ 확산 나서…
리트러스트, 블록체인 기반 ESG 서비스 확장… NFT·EU 규제 대응 솔루션 선보여
‘4살 된’ 토스뱅크, 회원 1375만명 돌파…혁신기술로 ‘포용금융’ 실천
카카오뱅크, 글로벌 인력 채용 본격화…태국·인니 진출 박차
'車보험 비교·추천 서비스 2.0' 계약체결률 약진
네·카·토 '선불충전금'도 상속 가능
3분기 순익 5조 육박…4대 금융, '역대급 실적' 전망
‘항암보장’ vs ‘라이브청구’…펫보험 경쟁 2라운드 돌입
외국인도 현지서 국내주식 사고 판다… 하나증권, 통합계좌 첫 거래 성사
은행대리업, 효율·포용 '두마리 토끼' 잡을까
2금융권 고금리 상품 실종에… 예금자보호 1억 상향에도 머니무브 ‘아직’
은행권 ISA수익률 ‘상위 1%’, 국내 주식 비중 9배 껑충
손해율 악화·비용 부담…보험업계 3분기도 막막
39만명 2

# 날짜별 리스트 + 뉴스모음 코드 합치기

In [ ]:
import pandas as pd
import requests
import re
import time
from bs4 import BeautifulSoup as bs

In [79]:
"핀테크 주요뉴스" in soup.select("a.txtl")[7].text

True

In [80]:
def fintech_news_list():
    url1 = "https://fintech.or.kr/web/board/boardContentsList.do"
    payload1 = dict(miv_pageNo=1, mode="W", board_id=6)
    r1 = requests.post(url1, data=payload1)
    soup = bs(r1.content, "lxml")
    contents_id_list = []
    for date, content_list in zip(soup.select("td.last"), soup.select("a.txtl")):
        if "핀테크 주요뉴스" in content_list.text:
            date = text_clean(date.text)
            contents_id = content_list['href'].split("'")[1]
            contents_id_list.append((date, contents_id))
    return contents_id_list

In [81]:
fintech_news_list()

[('2025-10-21', 'f61663288d694d068e33d00054b22595'),
 ('2025-10-20', '760c4ae4b76f449aa23de004e8200bc5'),
 ('2025-10-17', 'ad2eb2c521ab4e9fa32cb00a625cbd69'),
 ('2025-10-16', 'c80032073f994fb4b669d70bb5917c18'),
 ('2025-10-15', 'c579f3baf033478483098dc813b47e1f'),
 ('2025-10-14', 'f7a7f75aa51045209c20eff82dc21e90'),
 ('2025-10-13', 'b45fb58dc8094571a48344594ab19176'),
 ('2025-10-10', 'aeea59af5aea4346a8d53ed2605993e5')]

In [82]:
result = {}
contents_id_list = fintech_news_list()
for idx, (date, content_id) in enumerate(contents_id_list):
    print(f"{idx+1}/{len(contents_id_list)} {content_id} 수집중", end="\r")
    url2 = "https://fintech.or.kr/web/board/boardContentsView.do"
    payload2 = dict(miv_pageNo=1, mode="W", contents_id=content_id, board_id=6, searchkey="I")
    r2 = requests.post(url2, data=payload2)
    soup2 = bs(r2.content, "lxml")
    
    for td in soup2.select("tbody td"):
        if td.select_one("a") != None and td.select_one("a")['href'] != "https://fintech.or.kr/":
    #         print(td.select_one("a").text)
    #         print(td.select_one("a")['href'])
            result.setdefault("제목", []).append(td.select_one("a").text)
            result.setdefault("원문링크", []).append(td.select_one("a")['href'])
df = pd.DataFrame(result)

In [83]:
df

,제목,원문링크
0,"한국핀테크지원센터, 핀에듀 AI 활용 실무 온라인 교육 과정 5종 신규 개설",https://www.etnews.com/20251017000242
1,[한국핀테크지원센터] 2025 핀테크 전문가(개발자) 과정 ‘FinBoost Aca...,https://fintech.or.kr/web/board/boardContentsV...
2,[한국핀테크지원센터] AICC 기반 금융 서비스 자동화 교육 과정 수강생 모집(~선착순),https://fintech.or.kr/web/board/boardContentsV...
3,[한국핀테크지원센터] (온라인) 핀테크를 통한 금융 AI 트렌드와 혁신 사례 교육생...,https://fintech.or.kr/web/board/boardContentsV...
4,[한국핀테크지원센터] 2025년 핀테크 통번역존 운영(~예산 소진 시까지),https://fintech.or.kr/web/board/boardContentsV...
...,...,...
401,"美 셧다운·금리인하 기조에… 금·은·코인으로, 돈의 대이동",https://n.news.naver.com/mnews/article/008/000...
402,"“AI붐은 버블, 25년전 닷컴 때과 비슷”...IMF·영란은행, 증시 과열에 경고",https://n.news.naver.com/mnews/article/009/000...
403,[디캠프·한국핀테크지원센터] startup OI #금융권-스타트업 우수 협력 사례 ...,https://fintech.or.kr/web/board/boardContentsV...
404,[금융보안원] 「데이터허브」 기업 무료 이용 안내(상시),https://fintech.or.kr/web/board/boardContentsV...


In [56]:
from datetime import datetime

In [63]:
today = datetime.today()
formatted = today.strftime("%Y-%m-%d")
print(formatted)

2025-10-21


In [64]:
formatted == "2025-10-21"

True